Convert CARDIO:DE UIMA CAS XMI into Huggingface Dataset and split CARDIODE400 into three-fold split.

In [ ]:
from collections import Counter
from pathlib import Path
from importlib.metadata import version
from io import BytesIO
import hashlib
import json
import platform
from zipfile import ZipFile

from cassis import load_cas_from_xmi, load_typesystem
from datasets import Dataset, DatasetDict, Features, Value, load_from_disk
from tqdm.auto import tqdm

from sklearn.model_selection import KFold

In [ ]:
# Folder containing cas/CARDIODE400_main files.
CORPUS_ROOT = Path("..").resolve()

MAIN_PATH = CORPUS_ROOT / "cas" / "CARDIODE400_main"
OUTPUT_ROOT = Path("../../data/cardiode_hf_cv3").resolve()

LAYER_NAME = "webanno.custom.CHA2DS2_VA"
TYPESYSTEM_PATH = None  # Optional corpus-local typesystem.xml; otherwise discover nearby.
CAS_VIEW = "_InitialView"

In [28]:
# Find all CARDIO:DE400 letters

def letter_id(path):
    return next(name for name in [path.stem, path.parent.stem] if name.isdecimal())

sources = sorted(
    (p for p in MAIN_PATH.rglob("*") if p.is_file() and p.suffix.lower() in {".xmi", ".zip"}),
    key=lambda p: int(letter_id(p)),
)
ids = [letter_id(p) for p in sources]
assert len(ids) == len(set(map(int, ids))) == 400, "Expected 400 unique main letters."

In [ ]:
# Load XMI and typesystems, convert main annotations

def load_document(path):
    source = str(path.relative_to(CORPUS_ROOT))
    ts_bytes = None
    if path.suffix.lower() == ".zip":
        with ZipFile(path) as archive:
            xmi_name, = [n for n in archive.namelist() if n.lower().endswith(".xmi")]
            xmi_bytes = archive.read(xmi_name)
            source += "::" + xmi_name
            ts_names = [n for n in archive.namelist() if Path(n).name.lower() == "typesystem.xml"]
            if ts_names:
                ts_name, = ts_names
                ts_bytes = archive.read(ts_name)
    else:
        xmi_bytes = path.read_bytes()

    if TYPESYSTEM_PATH is not None:
        ts_bytes = Path(TYPESYSTEM_PATH).read_bytes()
    elif ts_bytes is None:
        ts_path = next(
            p for folder in path.parents
            if folder == CORPUS_ROOT or CORPUS_ROOT in folder.parents
            for p in folder.iterdir() if p.is_file() and p.name.lower() == "typesystem.xml"
        )
        ts_bytes = ts_path.read_bytes()
    cas = load_cas_from_xmi(BytesIO(xmi_bytes), typesystem=load_typesystem(BytesIO(ts_bytes)))
    hashes = {"xmi_sha256": hashlib.sha256(xmi_bytes).hexdigest(),
              "typesystem_sha256": hashlib.sha256(ts_bytes).hexdigest()}
    return cas.get_view(CAS_VIEW), source, hashes


def feature_value(value):
    if value is None or isinstance(value, (str, int, float, bool)):
        return value
    if isinstance(value, (list, tuple)):
        return [feature_value(v) for v in value]
    reference = {"type": value.type.name, "xmi_id": value.xmiID}
    if hasattr(value, "elements"):
        reference["elements"] = feature_value(value.elements)
    return reference


def convert(path):
    cas, source, hashes = load_document(path)
    selected = cas.select(LAYER_NAME) if LAYER_NAME else cas.select_all()
    annotations, final_score = [], None
    for annotation in selected:
        if annotation.type.name.startswith("uima.") or not hasattr(annotation, "begin"):
            continue
        names = {f.name for f in annotation.type.all_features} - {"begin", "end", "sofa"}
        values = {name: feature_value(annotation.get(name)) for name in sorted(names)}
        row = {"begin": annotation.begin, "end": annotation.end,
               "text": annotation.get_covered_text(), "annotation_type": annotation.type.name,
               "features_json": json.dumps(values, ensure_ascii=False, sort_keys=True)}
        for name in ("value", "polarity", "status", "trigger"):
            row[name] = None if values.get(name) is None else str(values[name])
        score = values.get("finalscore")
        row["final_score"] = None if score is None else str(score)
        annotations.append(row)
        if row["value"] == "FINAL_SCORE" and score is not None:
            final_score = float(str(score).replace(",", "."))
    return {"letter_id": letter_id(path), "xmi_path": source, "text": cas.sofa_string,
            "final_score": final_score, "annotations": annotations}, hashes

rows, input_hashes = [], {}
for path in tqdm(sources, desc="Convert main letters"):
    row, hashes = convert(path)
    rows.append(row)
    input_hashes[row["letter_id"]] = hashes

In [ ]:
# build Hugginface dataset

annotation_features = {
    "begin": Value("int64"), "end": Value("int64"),
    **{name: Value("string") for name in (
        "text", "value", "polarity", "status", "trigger", "final_score",
        "annotation_type", "features_json",
    )},
}
features = Features({
    "letter_id": Value("string"), "xmi_path": Value("string"), "text": Value("string"),
    "final_score": Value("float64"), "annotations": [annotation_features],
})
train_dataset = Dataset.from_list(rows, features=features)
print(train_dataset)

In [ ]:
# Prepare 3 folds

kfold = KFold(n_splits=3, shuffle=True, random_state=42)

fold_datasets = []
for number, (train_indices, test_indices) in enumerate(
    kfold.split(range(len(train_dataset))), start=1
):
    fold = DatasetDict({
        "train": train_dataset.select(train_indices.tolist()),
        "test": train_dataset.select(test_indices.tolist()),
    })
    fold_datasets.append(fold)
    print(f"Fold {number}: {len(fold['train'])} train, {len(fold['test'])} validation")

In [ ]:
# Write dataset to disk

OUTPUT_ROOT.mkdir(parents=True, exist_ok=False)
train_dataset.save_to_disk(OUTPUT_ROOT / "cha2ds2_va_train")
cv_root = OUTPUT_ROOT / "cha2ds2_va_CARDIODE400_cv3"
cv_root.mkdir()
for number, fold in enumerate(fold_datasets, start=1):
    fold.save_to_disk(cv_root / f"cha2ds2_va_fold_{number}")

manifest = {
    "algorithm": {"source": "KFold", "n_splits": 3, "shuffle": True, "random_state": 42,
                  "input_order": "numeric letter ID"},
    "letter_ids": ids,
    "folds": [{"fold": i, "train_ids": list(fold["train"]["letter_id"]),
               "test_ids": list(fold["test"]["letter_id"])}
              for i, fold in enumerate(fold_datasets, start=1)],
    "input_hashes": input_hashes,
    "annotation_layer": LAYER_NAME,
    "cas_view": CAS_VIEW,
    "environment": {"python": platform.python_version(),
                    **{name: version(name) for name in ("dkpro-cassis", "datasets", "tqdm")}},
}
(OUTPUT_ROOT / "split_manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False) + "\n", encoding="utf-8"
)
print(f"Saved datasets and manifest to {OUTPUT_ROOT}")
